# Jetson perf/ftrace 실습 복습 노트북

이 노트북은 `jetson-perf-profiling` 저장소에서 진행한 실습 전체를 순서대로 다시 실행하며 복습하기 위한 것입니다.
문서 버전은 [`docs/log.md`](../docs/log.md)(원본 로그), [`docs/시행착오_기록.md`](../docs/시행착오_기록.md)(막혔던 문제),
[`docs/반복측정_결과.md`](../docs/반복측정_결과.md)(반복 측정)를 참고하세요.

**주의**:
- 셀은 위에서 아래로 순서대로 실행하세요 (`Restart & Run All` 가능하도록 설계했지만, 일부 셀은 수 초 걸립니다).
- `sudo`가 필요한 셀이 많습니다 — 이 계정은 비밀번호 없이 sudo가 가능하도록 설정되어 있습니다.
- ⚠️ 표시된 셀은 **전역 커널 상태(ftrace)를 바꾸는 셀**입니다. 각 셀은 끝에 반드시 원복 코드를 포함합니다 — 중간에 실행을 멈추면 원복이 안 될 수 있으니 주의하세요.


## 0. 환경 확인

In [ ]:
%%bash
uname -a
nproc
free -h


## 1. perf 확보 — 벤더 커널 우회

Tegra 커널(`6.8.12-1021-tegra`)은 NVIDIA 커스텀 빌드라 대응하는 `linux-tools-6.8.12-1021-tegra` 패키지가
Ubuntu 표준 저장소에 없습니다. 아래 셀에서 재현합니다.

In [ ]:
%%bash
perf --version || true


우회: 버전이 다른 표준 `linux-tools-generic` 패키지의 perf 바이너리를 절대경로로 직접 호출합니다.

In [ ]:
%%bash
ls /usr/lib/linux-tools/*/perf
PERF=$(ls /usr/lib/linux-tools/*/perf | head -1)
echo "PERF=$PERF" > /tmp/perf_path.env
$PERF --version


## 2. 하드웨어 PMU 이벤트 확인

VM(VirtualBox)에서는 PMU 하드웨어 이벤트가 가상화 제약으로 `<not supported>`였는데,
실물 Jetson에서는 정상 수집되는지 확인합니다.

In [ ]:
%%bash
source /tmp/perf_path.env
sudo $PERF stat -- ls /


**기대 결과** (참고용 — 실제 실행값은 매번 조금씩 다릅니다):

```
task-clock   1.53 msec
cycles       1,612,215
instructions 1,468,134   # 0.91 insn per cycle
branches     304,389
branch-misses 15,552     # 5.11% of all branches
```


## 3. CPU 부하 예제 빌드 (Makefile)

In [ ]:
%%bash
cd ..
make -C src
ls -la src/hot src/io_bound src/multithread


## 4. perf record + Flame Graph

`fib(42)`를 `-O0`으로 컴파일(인라이닝 방지)해 재귀 호출 트리가 그대로 남도록 했습니다.

In [ ]:
%%bash
cd ..
time src/hot


In [ ]:
%%bash
cd ..
source /tmp/perf_path.env
sudo $PERF record -g -o /tmp/perf_review.data -- src/hot
sudo chown $(whoami) /tmp/perf_review.data
$PERF report -i /tmp/perf_review.data --stdio | head -30


FlameGraph 툴체인이 없으면 클론합니다 (`.gitignore` 처리되어 있어 매번 없을 수 있습니다).

In [ ]:
%%bash
cd ..
[ -d tools/FlameGraph ] || git clone --quiet https://github.com/brendangregg/FlameGraph.git tools/FlameGraph
source /tmp/perf_path.env
$PERF script -i /tmp/perf_review.data > /tmp/out_review.perf
tools/FlameGraph/stackcollapse-perf.pl /tmp/out_review.perf > /tmp/out_review.folded
tools/FlameGraph/flamegraph.pl /tmp/out_review.folded > /tmp/flame_review.svg
ls -la /tmp/flame_review.svg


저장된 결과물은 [`docs/flame_fib42.svg`](../docs/flame_fib42.svg)에 있습니다 (VS Code Explorer에서 열어보세요).

**해석**: `main`/`hot` 받침대 위로 `fib` 박스가 약 24층 쌓인 "뾰족탑" 모양. 위로 갈수록 폭이 좁아짐(100%→1.52%) —
`fib(n-1)`이 더 깊이 파고들고 `fib(n-2)`가 얕게 끝나는 비대칭 재귀 구조가 폭 변화로 시각화된 것.

## 5. perf stat 비교: `ls /` vs `fib(42)`

반복 측정(5회) 결과는 [`docs/반복측정_결과.md`](../docs/반복측정_결과.md)에 정리되어 있습니다.
여기서는 스크립트로 재현합니다.

In [ ]:
%%bash
cd ..
bash scripts/bench.sh 5


**핵심 숫자** (반복 측정 확정값):

| | `ls /` | `fib(42)` | `io_bound` | `multithread` |
|---|---|---|---|---|
| IPC | 0.91 | 3.37 | 1.91 | 3.36 |
| branch-miss | 5.11% | 0.01% | 0.02% | 0.01% |

`fib`처럼 분기 패턴이 규칙적인 루프는 분기 예측기가 거의 완벽히 학습되어 미스율이 0.01%까지 떨어짐.
`ls`는 초단명 프로세스라 예측기 워밍업 시간이 없어 미스율이 500배 가까이 높음.

## 6. ftrace — 벤더 커널 제약

`function`/`function_graph` tracer가 이 커널에 있는지 확인합니다 (Bootlin 과정 등에서 표준으로 다루는 tracer).

In [ ]:
%%bash
sudo cat /sys/kernel/debug/tracing/available_tracers
sudo cat /sys/kernel/debug/tracing/current_tracer


`nop`만 있으면 이 Tegra 커널은 `CONFIG_FUNCTION_TRACER`/`CONFIG_FUNCTION_GRAPH_TRACER`를 빼고 빌드된 것입니다.
대안으로 살아있는 tracepoint(`raw_syscalls`, `sched:sched_switch`)를 씁니다.

In [ ]:
%%bash
cd ..
sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/tracing_on'

src/hot

sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/tracing_on'
sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'

echo "=== hot 관련 줄 수 ==="
sudo grep -c 'hot' /sys/kernel/debug/tracing/trace
echo "=== 앞부분(런타임 초기화 syscall들) ==="
sudo grep 'hot' /sys/kernel/debug/tracing/trace | head -10
echo "=== 뒷부분(sched_switch로 선점된 흔적) ==="
sudo grep 'hot' /sys/kernel/debug/tracing/trace | tail -5

sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'


**중요 발견**: `perf stat`(sudo 없이)에서는 `context-switches:u 0`이었는데 ftrace에는 `sched_switch`가
찍혀 있었습니다. 모순이 아니라 `perf stat`의 `:u` 접미사가 user-space 이벤트만 카운트한다는 뜻입니다.
**도구마다 "무엇을 세는지"가 다르면 겉보기 모순이 생길 수 있습니다.**

## 7. I/O 바운드 워크로드 (`io_bound.c`)

`write()`를 200만 번 반복 — 연산은 거의 없고 커널 시간이 대부분.

In [ ]:
%%bash
cd ..
time src/io_bound


In [ ]:
%%bash
cd ..
source /tmp/perf_path.env
sudo $PERF stat -- src/io_bound


**주의**: `sudo` 없이 재면 `:u` 스코프라 커널에서 벌어지는 일(write 시스템콜 대부분)이 안 보입니다.
반드시 `sudo`로 재야 의미 있는 숫자가 나옵니다 (5절의 `io_bound` 열이 그 결과).

## 8. ⚠️ ftrace 버퍼 오버플로 재현 (선택 실행)

`write()` 200만 회를 기본 버퍼(1,410KB)로 추적하면 대부분이 잘려나갑니다.
**이 셀은 몇 초 걸리고 버퍼 크기를 일시적으로 늘립니다.** 필요 없으면 건너뛰어도 됩니다.

In [ ]:
%%bash
cd ..
echo "=== 기본 버퍼로 추적 (많이 잘릴 것) ==="
sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/tracing_on'
src/io_bound
sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/tracing_on'
sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'
echo "write(NR 64) 캡처 건수 (기대: 200만보다 훨씬 적음):"
sudo grep 'io_bound' /sys/kernel/debug/tracing/trace | awk '$5=="sys_enter:" && $7==64' | wc -l
sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'


In [ ]:
%%bash
cd ..
echo "=== 버퍼를 256MB로 키우고 재시도 ==="
free -h
sudo sh -c 'echo 262144 > /sys/kernel/debug/tracing/buffer_size_kb'

sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'
sudo sh -c 'echo 1 > /sys/kernel/debug/tracing/tracing_on'
IO_BOUND_OUT=/tmp/io_bound_review.bin src/io_bound
sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/tracing_on'
sudo sh -c 'echo 0 > /sys/kernel/debug/tracing/events/raw_syscalls/enable'

echo "write(NR 64) 캡처 건수 (기대: 200만에 근접):"
sudo grep 'io_bound' /sys/kernel/debug/tracing/trace | awk '$5=="sys_enter:" && $7==64' | wc -l

# 반드시 원복
sudo sh -c 'echo 1410 > /sys/kernel/debug/tracing/buffer_size_kb'
sudo sh -c 'echo > /sys/kernel/debug/tracing/trace'
rm -f /tmp/io_bound_review.bin
free -h


## 9. 멀티스레드 워크로드

4 스레드가 각각 `fib(38)`을 계산 — 이번 실습에서 실제로 컨텍스트 스위치/코어 마이그레이션이 관찰되는 유일한 케이스입니다.

In [ ]:
%%bash
cd ..
time src/multithread


In [ ]:
%%bash
cd ..
source /tmp/perf_path.env
sudo $PERF stat -- src/multithread


**주의(반증된 결론)**: 처음엔 "멀티스레드에서 처음으로 context-switch가 0이 아니게 나왔다"고 결론 냈지만,
이건 단일 스레드를 `sudo` 없이(`:u` 스코프) 쟀기 때문이었습니다. `sudo`로 통일하면 단일 스레드도
context-switches 22 / cpu-migrations 2가 나옵니다. 멀티스레드의 진짜 구별점은 **`CPUs utilized` ≈ 3.9**뿐입니다.

## 10. perf annotate — 소스라인 단위 분석

In [ ]:
%%bash
cd ..
source /tmp/perf_path.env
sudo $PERF record -g -o /tmp/perf_annotate_review.data -- src/hot
sudo chown $(whoami) /tmp/perf_annotate_review.data
$PERF annotate -i /tmp/perf_annotate_review.data --stdio fib 2>&1 | head -40


**한계 두 가지**:
1. `fib(n-1)`과 `fib(n-2)` 호출부는 같은 함수 본문을 실행하므로, annotate로는 "어느 가지가 더 뜨거운가"를 답할 수 없습니다.
   그건 Flame Graph(호출 구조)의 역할이고, annotate는 명령어 단위 핫맵이라 서로 다른 질문에 답합니다.
2. `cmp` 같은 1사이클 명령어가 비정상적으로 높은 퍼센트로 나올 수 있습니다 — ARM PMU의 샘플 스큐(skid) 때문에
   실제 정지 지점보다 몇 명령어 뒤에 귀속되는 현상입니다. **명령어별 퍼센트는 "이 근처가 핫스팟" 정도로 읽어야 합니다.**

## 11. 정리

| 발견 | 내용 |
|---|---|
| PMU | VM에서 안 되던 하드웨어 카운터가 실물에서는 전부 정상 |
| 워크로드별 차이 | 순간형/연산형/I·O형/병렬형에 따라 IPC·분기미스·컨텍스트스위치 패턴이 완전히 다름 |
| 도구 scope | `perf stat`과 `ftrace`가 다른 걸 셀 수 있다 (`:u` 스코프) |
| 벤더 커널 제약 | `perf` 패키지, `function_graph` tracer 모두 Tegra 커널에서 빠져 있음 |
| 자기 반증 | 반복 측정으로 스스로 낸 결론 2건을 반증 — 에러 체크 누락이 그럴듯한 거짓 숫자를 만들어냄 |
| 콜스택 검증 | 간접 추론(명령어 수 산술)에 그치지 않고 ftrace로 `write(-1,...) = -9 (EBADF)`를 직접 확인 |

더 자세한 내용은 [`README.md`](../README.md), [`docs/log.md`](../docs/log.md),
[`docs/시행착오_기록.md`](../docs/시행착오_기록.md), [`docs/반복측정_결과.md`](../docs/반복측정_결과.md),
[부록: x86 function_graph 비교](../docs/x86_function_graph_비교.md)를 참고하세요.